## Milvus Vector DB for RAG 

### Create a collection of legal docs

In [1]:
%pip install pymilvus


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv
# from pymilvus import connections

# # If using Docker standalone Milvus
# connections.connect("default", host="127.0.0.1", port="19530")

from pymilvus import connections

load_dotenv(override=True, dotenv_path="../.env.local")

milvus_uri = os.getenv("MILVUS_URI")
milvus_token = os.getenv("MILVUS_API_KEY")


connections.connect(
    alias="default",
    uri=milvus_uri,
    token=milvus_token
)

print("Connected to Milvus on Zilliz Cloud")


/var/folders/b8/45pl36ms0xx399j715gybz840000gn/T/ipykernel_80080/3686056658.py:16: PyMilvusDeprecationWarning: `connections.connect` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  connections.connect(


Connected to Milvus on Zilliz Cloud


In [ ]:
from pymilvus import db
from pymilvus import Collection, FieldSchema, CollectionSchema, DataType

# 1. Create a new database
# db.create_database("rag_db")

# 2. Switch to that database
db.using_database("rag_db")

# ----- Create schema -----
fields = [
    FieldSchema("doc_id", DataType.INT64, is_primary=True, auto_id=False),
    FieldSchema("title", DataType.VARCHAR, max_length=200),
    FieldSchema("domain", DataType.VARCHAR, max_length=100),
    FieldSchema("content", DataType.VARCHAR, max_length=2000),
    FieldSchema("embedding", DataType.FLOAT_VECTOR, dim=384) 
]

schema = CollectionSchema(fields, description="Policy documents with embeddings")

collection_name = "policy_docs_collection"
collection = Collection(collection_name, schema)

# ----- Create index -----
index_params = {
    "index_type": "IVF_FLAT",
    "metric_type": "COSINE",
    "params": {"nlist": 128},
}
collection.create_index(field_name="embedding", index_params=index_params)


/var/folders/b8/45pl36ms0xx399j715gybz840000gn/T/ipykernel_80080/2695431394.py:8: PyMilvusDeprecationWarning: `db.using_database` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  db.using_database("rag_db")
/var/folders/b8/45pl36ms0xx399j715gybz840000gn/T/ipykernel_80080/2695431394.py:22: PyMilvusDeprecationWarning: `Collection` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection = Collection(collection_name, schema)
/var/folders/b8/45pl36ms0xx399j715gybz840000gn/T/ipykernel_80080/2695431394.py:30: PyMilvusDeprecationWarning: `Collection.create_index` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection.create_index(field_name="embedding", index_params=index_params)


Status(code=0, message=)

In [14]:

# ----- Example data -----
content_chunks = [
    {
        "doc_id": 1,
        "section": "Pay Policies",
        "title": "Employee Pay Policy",
        "domain": "Human Resources",
        "content": "Employees are paid bi-weekly via direct deposit."
    },
    {
        "doc_id": 2,
        "section": "Leave of Absence",
        "title": "Leave Request and Approval Process",
        "domain": "Human Resources",
        "content": "Employees must submit a leave request for approval."
    },
    {
        "doc_id": 3,
        "section": "Internet Use",
        "title": "Acceptable Use of Company Internet",
        "domain": "IT & Security",
        "content": "Company internet must be used for work-related tasks only."
    },
    {
        "doc_id": 4,
        "section": "Break at Work",
        "title": "Employee Break Policy",
        "domain": "Workplace Operations",
        "content": "Employees can take an hour break."
    },
    {
        "doc_id": 5,
        "section": "Harassment",
        "title": "Workplace Harassment Policy",
        "domain": "Compliance",
        "content": "Interact with each employee with respect."
    }
]


# content_chunks_list = []
# for chunk in content_chunks:
#     content_chunks_list.append(chunk["content"])
content_chunks_list = [chunk["content"] for chunk in content_chunks]
print(content_chunks_list)


['Employees are paid bi-weekly via direct deposit.', 'Employees must submit a leave request for approval.', 'Company internet must be used for work-related tasks only.', 'Employees can take an hour break.', 'Interact with each employee with respect.']


In [15]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")

doc_vectors = model.encode(content_chunks_list)
doc_vectors.shape

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5948.88it/s]


(5, 384)

In [ ]:
# ---- Build columnar data ----
doc_ids = [int(i + 1) for i in range(len(content_chunks))]             # INT64
titles = [str(doc["title"]) for doc in content_chunks]                 # VARCHAR
domains = [str(doc["domain"]) for doc in content_chunks]               # VARCHAR
content = [str(doc["content"]) for doc in content_chunks]               # VARCHAR
embeddings = [list(map(float, vec)) for vec in doc_vectors]       # FLOAT_VECTOR(768)


# ---- Insert column-wise ----
collection.insert([doc_ids, titles, domains, content, embeddings])
# INSERT INTO policy_docs_collection (doc_id, title, domain, content, embedding) 
# VALUES (1, "Employee Pay Policy", "Human Resources", 
# "Employees are paid bi-weekly via direct deposit.", [0.1, 0.2, ..., 0.3])

collection.flush()

print(f"Successfully inserted {len(doc_ids)} documents into Milvus.")


In [16]:
#Load the collection before searching or querying
collection.load()
res = collection.query(expr="doc_id > 0", 
        output_fields=["doc_id", "title", "domain", "content", "embedding"])
print(res)


/var/folders/b8/45pl36ms0xx399j715gybz840000gn/T/ipykernel_80080/1248771852.py:2: PyMilvusDeprecationWarning: `Collection.load` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection.load()


data: ["{'doc_id': 1, 'title': 'Employee Pay Policy', 'domain': 'Human Resources', 'content': 'Employees are paid bi-weekly via direct deposit.', 'embedding': [0.02472514472901821, -0.009081516414880753, 0.03887128457427025, 0.03230158984661102, -0.05890548974275589, 0.039124615490436554, 0.02601799927651882, -0.043690863996744156, -0.019160084426403046, -1.172910629065882e-06, 0.013653445988893509, 0.004326773341745138, -0.053785599768161774, 0.006868935190141201, 0.007638241164386272, -0.06069644168019295, -0.037602223455905914, 0.007140935864299536, 0.13387171924114227, 0.0017353571020066738, -0.0030755847692489624, -0.07781296968460083, -0.08451441675424576, -0.0032163711730390787, 0.1338280886411667, -0.036435868591070175, 0.016922781243920326, 0.04861479252576828, -0.04028797149658203, 0.019676746800541878, -0.043712880462408066, 0.051511477679014206, -0.01629958115518093, -0.03905918821692467, -0.004453910049051046, 0.00047443583025597036, -0.03120519034564495, 0.048956222832202

/var/folders/b8/45pl36ms0xx399j715gybz840000gn/T/ipykernel_80080/1248771852.py:3: PyMilvusDeprecationWarning: `Collection.query` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  res = collection.query(expr="doc_id > 0",


In [17]:
# print(utility.has_collection("demo_collection"))

# # Get details about a specific collection
# # Get collection details
# collection = Collection("demo_collection")  # instantiate the collection object
print(collection.schema)                    # show the schema
print(collection.num_entities)              # number of entities
print(collection.description)               # optional

{'auto_id': False, 'description': 'Policy documents with embeddings', 'fields': [{'name': 'doc_id', 'description': '', 'type': <DataType.INT64: 5>, 'is_primary': True, 'auto_id': False}, {'name': 'title', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 200}}, {'name': 'domain', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 100}}, {'name': 'content', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 2000}}, {'name': 'embedding', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 384}}], 'enable_dynamic_field': False, 'enable_namespace': False}


/var/folders/b8/45pl36ms0xx399j715gybz840000gn/T/ipykernel_80080/2025431850.py:7: PyMilvusDeprecationWarning: `Collection.num_entities` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  print(collection.num_entities)              # number of entities


5
Policy documents with embeddings


## ONLINE PROCESSING

In [9]:
query = "What’s the leave policy?"
query_vector = model.encode([query])[0]
query_vector[:5]  # Show only first 5 values

array([0.0604582 , 0.03882853, 0.01172252, 0.0312459 , 0.12124224],
      dtype=float32)

In [ ]:
# Search for closest match only in the 'Human Resources' domain
results = collection.search(
    data=[query_vector],
    anns_field="embedding",
    param={"metric_type": "COSINE", "params": {"nprobe": 10}},
    limit=3,
    # expr='domain == "Human Resources"',
    output_fields=["doc_id", "title", "domain", "content"]
)

context_sring = ""
for res in results[0]:
    print(f"doc_id={res.entity.get('doc_id')}, "
          f"title={res.entity.get('title')}, "
          f"domain={res.entity.get('domain')}, "
          f"content={res.entity.get('content')}, "
          f"score={res.distance}")
    context_sring += f"\n -- \n {res.entity.get('content')} " # Append content to context string

print("\nContext String for RAG:\n", context_sring)    


/var/folders/b8/45pl36ms0xx399j715gybz840000gn/T/ipykernel_80080/1512635299.py:2: PyMilvusDeprecationWarning: `Collection.search` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  results = collection.search(


doc_id=2, title=Leave Request and Approval Process, domain=Human Resources, content=Employees must submit a leave request for approval., score=0.6198839545249939
doc_id=4, title=Employee Break Policy, domain=Workplace Operations, content=Employees can take an hour break., score=0.38792210817337036
doc_id=1, title=Employee Pay Policy, domain=Human Resources, content=Employees are paid bi-weekly via direct deposit., score=0.23205101490020752

Context String for RAG:
 
 -- 
 Employees must submit a leave request for approval. 
 -- 
 Employees can take an hour break. 
 -- 
 Employees are paid bi-weekly via direct deposit. 


In [11]:
from llm_utlity import ask_question_open_ai 

query = "What’s the leave policy?"
response = ask_question_open_ai(query, context_sring)
response


'Employees must submit a leave request for approval (i.e., leaves must be requested and approved).'

In [12]:
print(f"User query: {query}")
print(f"Context: {context_sring}")

print(f"\n\nOpen AI Response: {response}")

User query: What’s the leave policy?
Context: 
 -- 
 Employees must submit a leave request for approval. 
 -- 
 Employees can take an hour break. 
 -- 
 Employees are paid bi-weekly via direct deposit. 


Open AI Response: Employees must submit a leave request for approval (i.e., leaves must be requested and approved).


In [ ]:
# RAG Evaluation
# Context_Recall: Did the system retrieve all relevant documents?
# Context_Precision: What proportion of retrieved documents are relevant?

input = "What’s the leave policy?"
expected_context = "Employees are entitled to 20 days of paid leave annually."
actual_context = '''
 Employees must submit a leave request for approval. 
 -- 
 Employees are paid bi-weekly via direct deposit. 
 '''
#Context_Recall: 0%


input = "What’s the leave policy?"
expected_context = ["Employees are entitled to 20 days of paid leave annually."," They must submit a leave request for approval."]
actual_context = ["Employees must submit a leave request for approval. ", " Employees are paid bi-weekly via direct deposit. ", " Company internet must be used for work-related tasks only. "]
#Context_Recall: 1 out of 2 expected = 1/2 = 50%


input = "What’s the leave policy?"
expected_context = ["Employees are entitled to 20 days of paid leave annually."," They must submit a leave request for approval."]
actual_context = ["They are entitled to 20 days of paid leave in a year."," They must submit a leave request for approval."]
#Context_Recall: 2 out of 2 expected = 2/2 = 100%

In [ ]:
# Context_Precision: What proportion of retrieved documents are relevant?
input = "What’s the leave policy?"
expected_context = ["Employees are entitled to 20 days of paid leave annually."," They must submit a leave request for approval."]
actual_context = ["Employees must submit a leave request for approval. ", " Employees are paid bi-weekly via direct deposit. ", " Company internet must be used for work-related tasks only. "]
#Context_Precision: 1 out of 3 retrieved = 1/3 = 33%


input = "What’s the leave policy?"
expected_context = ["Employees are entitled to 20 days of paid leave annually."," They must submit a leave request for approval."]
actual_context = ["They are entitled to 20 days of paid leave in a year."," They must submit a leave request for approval.", " Company internet must be used for work-related tasks only. "]
#Context_Precision: 2 out of 3 retrieved = 2/3 = 67%

In [ ]:
##### RAG Evaluation
### Retrieval Metrics
# Context_Recall: Did the system retrieve all relevant documents?
# Context_Precision: What proportion of retrieved documents are relevant?

### Generative Metrics
# Faithfulness - Does my answer accurately reflect the retrieved context? (Is the answer faithful to the retrieved information?)
# Accuracy - Does the answer accurately address the user's question based on the retrieved context? (Is the answer correct given the retrieved information?)


# F1 = 2 * (Context_Precision * Context_Recall) / (Context_Precision + Context_Recall) -- NOT USED

In [ ]:
### Generative Metrics
# Accuracy

input = "What’s the leave policy?"
context = ["Employees are entitled to 20 days of paid leave annually."," They must submit a leave request for approval."]

#expected_llm_output = '''
#llm as a judge
groundtruth = ''' 
    Based on the provided context, the leave policy states that employees are entitled to 20 days of paid leave annually 
    and must submit a leave request for approval.'''

actual_llm_output = '''
    Based on the provided context, the leave policy states that employees must submit a leave request for approval. 
    No other details are given. If you have more of the policy, I can summarize that as well.'''

accuracy = 0.50


# Faithfulness - Is the generated output consistent with the provided context?
faithfulness = 0.50